In [1]:
import mpmath as mp
mp.mp.dps = 50

# Optional pretty table support
try:
    import pandas as pd
except Exception:
    pd = None

def I_numeric(l, u, plus=True):
    J = lambda n, x: mp.besselj(n, x)
    denom = 2 * (J(l, u)**2) * (u**2)
    integrand = lambda x: x * (J(l-1, x)**2 + (J(l+1, x)**2 if plus else -J(l+1, x)**2))
    return mp.quad(integrand, [0, u]) / denom

def I_plus_exact(l, u):
    J = lambda n, x: mp.besselj(n, x)
    Jl, Jlp1, Jlm1 = J(l,u), J(l+1,u), J(l-1,u)
    Jp = (Jlm1 - Jlp1)/2   # J_l'(u)
    return (u**2 - l**2)/(2*u**2) + Jp/(u*Jl) + (Jp**2)/(2*Jl**2)

def I_minus_exact(l, u):
    return l/(u**2)

def _fmt(x, sig=6):
    try:
        return f"{float(x):.{sig}e}"
    except Exception:
        return str(x)

def _fmt_rel(x):
    try:
        return f"{float(x):.3e}"
    except Exception:
        return str(x)

tests = [(l,u) for l in [0,1,2,3,5] for u in [0.5,1.0,2.5,5.0,7.2,10.0]]

rows = []
for l,u in tests:
    if abs(mp.besselj(l,u)) < mp.mpf('1e-18'):  # avoid divide-by-near-zero
        continue
    Inp = I_numeric(l,u,True)
    Inm = I_numeric(l,u,False)
    Ipe = I_plus_exact(l,u)
    Ime = I_minus_exact(l,u)
    relp = abs((Inp - Ipe)/Ipe) if Ipe != 0 else mp.nan
    relm = abs((Inm - Ime)/Ime) if Ime != 0 else abs(Inm)
    rows.append({
        "ell": l,
        "u": f"{float(u):.2f}",
        "I+ num": _fmt(Inp),
        "I+ exact": _fmt(Ipe),
        "I+ rel err": _fmt_rel(relp),
        "I- num": _fmt(Inm),
        "I- exact": _fmt(Ime),
        "I- rel err": _fmt_rel(relm),
    })

if rows:
    if pd is not None:
        df = pd.DataFrame(rows)
        try:
            from IPython.display import display
            display(df)
        except Exception:
            print(df.to_string(index=False))
    else:
        header = f"{'ell':>3} {'u':>6} {'I+ num':>14} {'I+ exact':>14} {'I+ rel err':>12} {'I- num':>14} {'I- exact':>14} {'I- rel err':>12}"
        print(header)
        print('-'*len(header))
        for r in rows:
            line = f"{int(r['ell']):3d} {r['u']:>6} {r['I+ num']:>14} {r['I+ exact']:>14} {r['I+ rel err']:>12} {r['I- num']:>14} {r['I- exact']:>14} {r['I- rel err']:>12}"
            print(line)

,ell,u,I+ num,I+ exact,I+ rel err,I- num,I- exact,I- rel err
0,0,0.50,1.701611e-02,1.701611e-02,9.817e-51,9.106361e-60,0.000000e+00,9.106e-60
1,0,1.00,9.027811e-02,9.027811e-02,3.701e-51,3.656381e-59,0.000000e+00,3.656e-59
2,0,2.50,5.738696e+01,5.738696e+01,0.000e+00,2.610514e-56,0.000000e+00,2.611e-56
3,0,5.00,1.832208e+00,1.832208e+00,0.000e+00,-4.312762e-58,0.000000e+00,4.313e-58
4,0,7.20,4.913777e-01,4.913777e-01,1.645e-17,1.787353e-58,0.000000e+00,1.787e-58
5,0,10.00,5.332993e-01,5.332993e-01,2.506e-51,-1.994514e-58,0.000000e+00,1.995e-58
6,1,0.50,4.002688e+00,4.002688e+00,0.000e+00,4.000000e+00,4.000000e+00,0.000e+00
7,1,1.00,1.011862e+00,1.011862e+00,0.000e+00,1.000000e+00,1.000000e+00,0.000e+00
8,1,2.50,3.447369e-01,3.447369e-01,4.509e-17,1.600000e-01,1.600000e-01,2.082e-17
9,1,5.00,6.069629e-01,6.069629e-01,2.927e-17,4.000000e-02,4.000000e-02,2.082e-17


In [2]:
import mpmath as mp
mp.mp.dps = 50

# Optional pretty table support
try:
    import pandas as pd
except Exception:
    pd = None

def I_numeric_K(l, w, plus=True):
    K = lambda n, x: mp.besselk(n, x)
    denom = 2 * (K(l, w)**2) * (w**2)
    integrand = lambda y: y * ((K(l-1, y)**2) + (K(l+1, y)**2 if plus else -(K(l+1, y)**2)))
    val = mp.quad(integrand, [w, mp.inf])
    return val/denom

def I_analytic_forms(l, w):
    K = lambda n, x: mp.besselk(n, x)
    Kl = K(l, w)
    Kp = -0.5*(K(l-1, w) + K(l+1, w))  # derivative identity: K'_nu = -1/2(K_{ν-1}+K_{ν+1})
    Iminus = -l/(w**2)
    Iplus = (w**2 + l**2)/(2*w**2) - Kp/(w*Kl) - (Kp**2)/(2*Kl**2)
    return Iplus, Iminus

def _fmt(x, sig=6):
    try:
        return f"{float(x):.{sig}e}"
    except Exception:
        return str(x)

def _fmt_rel(x):
    try:
        return f"{float(x):.3e}"
    except Exception:
        return str(x)

rows = []
for l in [0,1,2,3,5]:
    for w in [0.5,1.0,2.5,5.0,10.0]:
        Inp = I_numeric_K(l, w, True)
        Inm = I_numeric_K(l, w, False)
        Ipe, Ime = I_analytic_forms(l, w)
        relp = abs((Inp - Ipe)/Ipe) if Ipe != 0 else abs(Inp - Ipe)
        relm = abs((Inm - Ime)/Ime) if Ime != 0 else abs(Inm - Ime)
        rows.append({
            "ell": l,
            "w": f"{float(w):.2f}",
            "I+ num": _fmt(Inp),
            "I+ exact": _fmt(Ipe),
            "I+ rel err": _fmt_rel(relp),
            "I- num": _fmt(Inm),
            "I- exact": _fmt(Ime),
            "I- rel err": _fmt_rel(relm),
        })

if rows:
    if pd is not None:
        df = pd.DataFrame(rows)
        try:
            from IPython.display import display
            display(df)
        except Exception:
            print(df.to_string(index=False))
    else:
        header = f"{'ell':>3} {'w':>6} {'I+ num':>14} {'I+ exact':>14} {'I+ rel err':>12} {'I- num':>14} {'I- exact':>14} {'I- rel err':>12}"
        print(header)
        print('-'*len(header))
        for r in rows:
            line = f"{int(r['ell']):3d} {r['w']:>6} {r['I+ num']:>14} {r['I+ exact']:>14} {r['I+ rel err']:>12} {r['I- num']:>14} {r['I- exact']:>14} {r['I- rel err']:>12}"
            print(line)

,ell,w,I+ num,I+ exact,I+ rel err,I- num,I- exact,I- rel err
0,0,0.50,2.478341e+00,2.478341e+00,0.000e+00,0.000000e+00,0.000000e+00,0.000e+00
1,0,1.00,9.077110e-01,9.077110e-01,2.945e-51,0.000000e+00,0.000000e+00,0.000e+00
2,0,2.50,2.717746e-01,2.717746e-01,4.917e-51,0.000000e+00,0.000000e+00,0.000e+00
3,0,5.00,1.187935e-01,1.187935e-01,1.687e-50,0.000000e+00,0.000000e+00,0.000e+00
4,0,10.00,5.483356e-02,5.483356e-02,1.676e-50,0.000000e+00,0.000000e+00,0.000e+00
5,1,0.50,4.344276e+00,4.344276e+00,4.922e-51,-4.000000e+00,-4.000000e+00,2.673e-51
6,1,1.00,1.255361e+00,1.255361e+00,0.000e+00,-1.000000e+00,-1.000000e+00,2.673e-51
7,1,2.50,3.040181e-01,3.040181e-01,1.315e-16,-1.600000e-01,-1.600000e-01,2.082e-17
8,1,5.00,1.235842e-01,1.235842e-01,1.437e-16,-4.000000e-02,-4.000000e-02,2.082e-17
9,1,10.00,5.549777e-02,5.549777e-02,8.002e-17,-1.000000e-02,-1.000000e-02,2.082e-17
